# PAC-3310 cAMP Analysis — Consolidated

This notebook analyzes PAC-3310 and carbachol concentration-response experiments in **CHO-M2** and **CHO-M4** cells. It includes six ELISA plates: carbachol agonism, PAC-3310 agonism, and PAC-3310 antagonism of a fixed carbachol challenge for each receptor.

The antagonist challenges are **20 nM carbachol for M2** and **100 nM carbachol for M4**. All reported sample concentrations are corrected for the **5× assay dilution**.

## 1. Setup

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import curve_fit
from scipy.stats import f as f_dist
from scipy.stats import t as t_dist
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
%matplotlib inline

plt.rcParams.update({
    'figure.dpi': 110,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'DejaVu Sans'],
    'mathtext.default': 'regular',
})

DILUTION_FACTOR = 5
LABEL_SIZE = 14
TICK_SIZE = 11
LEGEND_SIZE = 10
RECEPTOR_COLORS = {'M2': '#377eb8', 'M4': '#984ea3'}

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
DATA_ROOT = PROJECT_ROOT / 'data' / 'camp_assay'
RAW_DIR = DATA_ROOT / 'raw'
PROCESSED_DIR = DATA_ROOT / 'processed'
FIGURE_DIR = PROJECT_ROOT / 'figures' / 'camp_assay'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

STANDARD_CONCENTRATIONS = {
    'S2': 250, 'S3': 83.3, 'S4': 27.8,
    'S5': 9.3, 'S6': 3.1, 'S7': 1,
}
STANDARD_LABEL_MAP = {
    'Standard: 250 pmol/mL': 'S2',
    'Standard: 83.3 pmol/mL': 'S3',
    'Standard: 27.8 pmol/mL': 'S4',
    'Standard: 9.3 pmol/mL': 'S5',
    'Standard: 3.1 pmol/mL': 'S6',
    'Standard: 1 pmol/mL': 'S7',
}

## 2. Dataset Definitions

In [ ]:
DATASETS = {
    'Carbachol dose titration (M2)': {
        'file': 'CHO-M2_carbachol_cAMP.csv',
        'receptor': 'M2', 'mode': 'Carbachol agonism', 'compound': 'Carbachol',
        'conc_col': 'Carbachol Concentration (log M)',
        'controls': ['Vehicle Control'], 'fixed_carbachol_nm': np.nan,
    },
    'PAC-3310 dose titration (M2)': {
        'file': 'CHO-M2_PAC-3310_cAMP.csv',
        'receptor': 'M2', 'mode': 'PAC-3310 agonism', 'compound': 'PAC-3310',
        'conc_col': 'PAC-3310 Concentration (log M)',
        'controls': ['Vehicle Control'], 'fixed_carbachol_nm': np.nan,
    },
    'PAC-3310 + 20 nM carbachol (M2)': {
        'file': 'CHO-M2_PAC-3310_plus_20nM_carbachol_cAMP.csv',
        'receptor': 'M2', 'mode': 'PAC-3310 antagonism', 'compound': 'PAC-3310',
        'conc_col': 'PAC-3310 Concentration (log M)',
        'controls': ['Carbachol-only Control (20 nM)', 'Vehicle Control'],
        'fixed_carbachol_nm': 20,
    },
    'Carbachol dose titration (M4)': {
        'file': 'CHO-M4_carbachol_cAMP.csv',
        'receptor': 'M4', 'mode': 'Carbachol agonism', 'compound': 'Carbachol',
        'conc_col': 'Carbachol Concentration (log M)',
        'controls': ['Vehicle Control'], 'fixed_carbachol_nm': np.nan,
    },
    'PAC-3310 dose titration (M4)': {
        'file': 'CHO-M4_PAC-3310_cAMP.csv',
        'receptor': 'M4', 'mode': 'PAC-3310 agonism', 'compound': 'PAC-3310',
        'conc_col': 'PAC-3310 Concentration (log M)',
        'controls': ['Vehicle Control'], 'fixed_carbachol_nm': np.nan,
    },
    'PAC-3310 + 100 nM carbachol (M4)': {
        'file': 'CHO-M4_PAC-3310_plus_100nM_carbachol_cAMP.csv',
        'receptor': 'M4', 'mode': 'PAC-3310 antagonism', 'compound': 'PAC-3310',
        'conc_col': 'PAC-3310 Concentration (log M)',
        'controls': ['Carbachol-only Control (100 nM)', 'Vehicle Control'],
        'fixed_carbachol_nm': 100,
    },
}


## 3. Row-corrected ELISA fluorescence

The plate-reader export applied its selected **row correction before export**: for sample wells, each row median was shifted to the plate-wide sample median. Standard wells were not shifted. The CSV fluorescence values loaded below are therefore the corrected values used by the exported analyses.

This notebook preserves those values exactly and **does not apply the row correction a second time**.

In [ ]:
all_raw = {}
all_samples = {}
all_standards = {}

for condition, info in DATASETS.items():
    df = pd.read_csv(RAW_DIR / info['file'])
    df['Fluorescence'] = pd.to_numeric(df['Fluorescence'], errors='coerce')
    conc_col = info['conc_col']
    std_mask = df[conc_col].astype(str).str.startswith('Standard:')
    all_raw[condition] = df
    all_standards[condition] = df[std_mask].copy()
    all_samples[condition] = df[~std_mask].copy()

    numeric = pd.to_numeric(all_samples[condition][conc_col], errors='coerce')
    dose_counts = all_samples[condition][numeric.notna()].groupby(conc_col)['Replicate'].count()
    controls = all_samples[condition][numeric.isna()].groupby(conc_col)['Replicate'].count()
    print(condition)
    print(f'  standards: {std_mask.sum()}')
    print(f'  doses: {sorted(numeric.dropna().unique())}')
    print(f'  replicates per dose: {sorted(dose_counts.unique())}')
    print(f'  controls: {controls.to_dict()}')
    assert len(dose_counts) == 7 and set(map(float, dose_counts.index)) == set([-11., -10., -9., -8., -7., -6., -5.])
    assert dose_counts.eq(8).all()
    assert controls.eq(8).all()
    print()

## 4. Plate Layouts

In [ ]:
def parse_well(well):
    return ord(well[0].upper()) - ord('A'), int(well[1:]) - 1


def plot_plate(ax, condition):
    info = DATASETS[condition]
    df = all_raw[condition]
    conc_col = info['conc_col']
    plate = np.full((8, 12), np.nan)
    labels = np.full((8, 12), '', dtype=object)
    for _, row in df.iterrows():
        r, c = parse_well(row['Well'])
        plate[r, c] = row['Fluorescence']
        label = str(row[conc_col]).replace('Standard: ', '').replace(' pmol/mL', '')
        labels[r, c] = label[:9]

    im = ax.imshow(plate, cmap='YlOrRd', aspect='equal')
    plt.colorbar(im, ax=ax, shrink=0.72).set_label('Corrected fluorescence', fontsize=8)
    for r in range(8):
        for c in range(12):
            if np.isfinite(plate[r, c]):
                ax.text(c, r - 0.15, f'{plate[r, c]:.3f}', ha='center', va='center', fontsize=5.2, fontweight='bold')
                ax.text(c, r + 0.19, labels[r, c], ha='center', va='center', fontsize=4.4)
            else:
                ax.text(c, r, '—', ha='center', va='center', fontsize=7)
    ax.set_xticks(range(12), labels=range(1, 13), fontsize=7)
    ax.set_yticks(range(8), labels=list('ABCDEFGH'), fontsize=7)
    ax.set_xlabel('Column', fontsize=9)
    ax.set_ylabel('Row', fontsize=9)
    ax.set_title(condition, fontsize=10, fontweight='bold')
    ax.set_xticks(np.arange(-0.5, 12, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, 8, 1), minor=True)
    ax.grid(which='minor', color='white', linewidth=1)


fig, axes = plt.subplots(2, 3, figsize=(19, 9))
for ax, condition in zip(axes.flat, DATASETS):
    plot_plate(ax, condition)
plt.tight_layout()
plate_figure = FIGURE_DIR / 'PAC-3310_cAMP_plate_layouts.png'
plt.savefig(plate_figure, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {plate_figure}')

## 5. Standard Curve Analysis

Each plate is calibrated independently with the logit-log method:

$$B/B_0=\frac{F_{sample}-F_{NSB}}{F_{B_0}-F_{NSB}}$$

The logit of $B/B_0$ is regressed against $\log_{10}[\mathrm{cAMP}]$. Each standard concentration has one well, matching the plate configuration.

In [ ]:
def logit(x):
    return np.log(x / (1 - x))


def inv_logit(y):
    return np.exp(y) / (1 + np.exp(y))


def fit_standard_curve(standards_df, conc_col):
    nsb = standards_df.loc[standards_df[conc_col].eq('Standard: NSB'), 'Fluorescence'].mean()
    b0 = standards_df.loc[standards_df[conc_col].eq('Standard: B_0'), 'Fluorescence'].mean()
    corrected_b0 = b0 - nsb
    points = []
    for label, key in STANDARD_LABEL_MAP.items():
        values = standards_df.loc[standards_df[conc_col].eq(label), 'Fluorescence'].dropna()
        if len(values):
            fluorescence = values.mean()
            ratio = (fluorescence - nsb) / corrected_b0
            points.append({
                'Standard': key, 'Concentration (pmol/mL)': STANDARD_CONCENTRATIONS[key],
                'Fluorescence': fluorescence, 'B/B0': ratio, '%B/B0': ratio * 100,
            })
    points_df = pd.DataFrame(points)
    valid_df = points_df[points_df['B/B0'].between(0.01, 0.99, inclusive='neither')].copy()
    x = np.log10(valid_df['Concentration (pmol/mL)'].to_numpy(float))
    y = logit(valid_df['B/B0'].to_numpy(float))
    result = stats.linregress(x, y)
    return {
        'nsb': nsb, 'b0': b0, 'corrected_b0': corrected_b0,
        'slope': result.slope, 'intercept': result.intercept,
        'r_squared': result.rvalue ** 2, 'points': points_df, 'valid': valid_df,
    }


standard_curves = {}
standard_summary_records = []
for condition, info in DATASETS.items():
    params = fit_standard_curve(all_standards[condition], info['conc_col'])
    standard_curves[condition] = params
    standard_summary_records.append({
        'Condition': condition, 'Receptor': info['receptor'], 'Assay Mode': info['mode'],
        'NSB Fluorescence': params['nsb'], 'B0 Fluorescence': params['b0'],
        'Corrected B0': params['corrected_b0'], 'Slope': params['slope'],
        'Intercept': params['intercept'], 'R²': params['r_squared'],
        'Valid Standard Points': len(params['valid']), 'Total Standard Points': len(params['points']),
    })

standard_summary_df = pd.DataFrame(standard_summary_records)
display(standard_summary_df.round(4))

In [ ]:
def plot_standard_curve(ax, condition):
    p = standard_curves[condition]
    points = p['points']
    ax.scatter(points['Concentration (pmol/mL)'], points['%B/B0'], s=55,
               color=RECEPTOR_COLORS[DATASETS[condition]['receptor']],
               edgecolors='white', linewidth=0.8, zorder=5)
    x = np.logspace(-1, 3, 200)
    ax.plot(x, inv_logit(p['slope'] * np.log10(x) + p['intercept']) * 100,
            color='0.25', linewidth=1.8, label=f"R²={p['r_squared']:.3f}")
    ax.set_xscale('log')
    ax.set_xlim(0.1, 1000)
    ax.set_ylim(0, 100)
    ax.set_xlabel('cAMP concentration (pmol/mL)', fontsize=9)
    ax.set_ylabel('%B/B₀', fontsize=9)
    ax.set_title(condition, fontsize=9.5, fontweight='bold')
    ax.legend(frameon=False, fontsize=8)
    ax.grid(True, alpha=0.25)


fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, condition in zip(axes.flat, DATASETS):
    plot_standard_curve(ax, condition)
plt.tight_layout()
standard_figure = FIGURE_DIR / 'PAC-3310_cAMP_standard_curves.png'
plt.savefig(standard_figure, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {standard_figure}')

## 6. Back-calculated, Dilution-corrected cAMP

Each corrected sample fluorescence value is converted through its own plate's standard curve. The resulting measured concentration is multiplied by **5** so all downstream results describe the undiluted biological sample.

In [ ]:
def calculate_concentration(fluorescence, params):
    ratio = (fluorescence - params['nsb']) / params['corrected_b0']
    ratio_clamped = np.clip(ratio, 0.001, 0.999)
    log_concentration = (logit(ratio_clamped) - params['intercept']) / params['slope']
    if log_concentration < -2 or log_concentration > 4:
        return np.nan, ratio
    return 10 ** log_concentration, ratio


replicate_records = []
for condition, info in DATASETS.items():
    params = standard_curves[condition]
    conc_col = info['conc_col']
    for _, row in all_samples[condition].iterrows():
        label = str(row[conc_col])
        numeric_conc = pd.to_numeric(pd.Series([label]), errors='coerce').iloc[0]
        measured, ratio = calculate_concentration(row['Fluorescence'], params)
        replicate_records.append({
            'Condition': condition, 'Receptor': info['receptor'], 'Assay Mode': info['mode'],
            'Fixed Carbachol (nM)': info['fixed_carbachol_nm'], 'Compound': info['compound'],
            'Well': row['Well'], 'Plate Row': row['Well'][0], 'Replicate': int(row['Replicate']),
            'Condition Label': label, 'Concentration (log M)': numeric_conc,
            'Concentration (M)': 10 ** numeric_conc if pd.notna(numeric_conc) else np.nan,
            'Corrected Fluorescence': row['Fluorescence'], 'B/B0': ratio,
            'Measured cAMP (pmol/mL)': measured,
            'Real-sample cAMP (pmol/mL)': measured * DILUTION_FACTOR,
            'Standard-curve Linear Range Flag': 'Within 20–80% B/B0' if 0.2 <= ratio <= 0.8 else 'Outside 20–80% B/B0',
            'Is Control': label in info['controls'],
        })

replicate_df = pd.DataFrame(replicate_records)
display(replicate_df.head())
print(f"Valid back-calculations: {replicate_df['Real-sample cAMP (pmol/mL)'].notna().sum()} / {len(replicate_df)}")

## 7. Biological Response Normalization

For carbachol and PAC-3310 agonism, cAMP is expressed as a percentage of the matching plate's mean vehicle control:

$$\%\,\mathrm{vehicle}=100\frac{C_{sample}}{\overline{C}_{vehicle}}$$

For antagonism, the carbachol-only control defines 0% inhibition and the vehicle-only control defines 100% inhibition:

$$\%\,\mathrm{carbachol\ signaling\ inhibition}=100\frac{C_{sample}-\overline{C}_{carbachol}}{\overline{C}_{vehicle}-\overline{C}_{carbachol}}$$

These biological controls are applied after the row-corrected fluorescence values have been back-calculated to cAMP.

In [ ]:
replicate_df['Normalized Response'] = np.nan
replicate_df['Normalized Endpoint'] = ''
control_summary_records = []

for condition, info in DATASETS.items():
    mask = replicate_df['Condition'].eq(condition)
    controls = replicate_df[mask & replicate_df['Is Control']]
    for label, group in controls.groupby('Condition Label'):
        control_summary_records.append({
            'Condition': condition, 'Receptor': info['receptor'], 'Assay Mode': info['mode'],
            'Control': label, 'Mean cAMP (pmol/mL)': group['Real-sample cAMP (pmol/mL)'].mean(),
            'SEM cAMP (pmol/mL)': group['Real-sample cAMP (pmol/mL)'].sem(), 'n': len(group),
        })

    if info['mode'] == 'PAC-3310 antagonism':
        carbachol_label = next(x for x in info['controls'] if x.startswith('Carbachol-only'))
        carbachol_mean = controls.loc[controls['Condition Label'].eq(carbachol_label), 'Real-sample cAMP (pmol/mL)'].mean()
        vehicle_mean = controls.loc[controls['Condition Label'].eq('Vehicle Control'), 'Real-sample cAMP (pmol/mL)'].mean()
        replicate_df.loc[mask, 'Normalized Response'] = 100 * (
            replicate_df.loc[mask, 'Real-sample cAMP (pmol/mL)'] - carbachol_mean
        ) / (vehicle_mean - carbachol_mean)
        replicate_df.loc[mask, 'Normalized Endpoint'] = '% Carbachol Signaling Inhibition'
    else:
        vehicle_mean = controls.loc[controls['Condition Label'].eq('Vehicle Control'), 'Real-sample cAMP (pmol/mL)'].mean()
        replicate_df.loc[mask, 'Normalized Response'] = (
            100 * replicate_df.loc[mask, 'Real-sample cAMP (pmol/mL)'] / vehicle_mean
        )
        replicate_df.loc[mask, 'Normalized Endpoint'] = 'cAMP (% Vehicle Control)'

control_summary_df = pd.DataFrame(control_summary_records)
crc_df = replicate_df[(~replicate_df['Is Control']) & replicate_df['Concentration (log M)'].notna()].copy()
group_summary_df = (
    crc_df.groupby(['Condition', 'Receptor', 'Assay Mode', 'Fixed Carbachol (nM)',
                    'Normalized Endpoint', 'Concentration (log M)'], dropna=False)
    .agg(Mean_cAMP_pmol_mL=('Real-sample cAMP (pmol/mL)', 'mean'),
         SEM_cAMP_pmol_mL=('Real-sample cAMP (pmol/mL)', 'sem'),
         Mean_Normalized_Response=('Normalized Response', 'mean'),
         SEM_Normalized_Response=('Normalized Response', 'sem'),
         n=('Normalized Response', 'count'))
    .reset_index()
)

display(control_summary_df.round(3))
display(group_summary_df.round(3))

## 8. 4PL Curve Fitting and Model Selection

Replicate-level normalized responses are fitted with a four-parameter logistic model. An extra sum-of-squares F-test compares the 4PL model with a flat response. A curve and potency estimate are reported only when the 4PL model is supported at $p<0.01$, the midpoint lies within the tested concentration range, and the fitted direction matches the assay mode (decreasing cAMP for agonism; increasing inhibition for antagonism).

In [ ]:
def model_4pl(log_conc, bottom_at_low, hill, log_ec50, top_at_high):
    return bottom_at_low + (top_at_high - bottom_at_low) / (1 + 10 ** ((log_ec50 - log_conc) * hill))


def fit_4pl(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    keep = np.isfinite(x) & np.isfinite(y)
    x, y = x[keep], y[keep]
    if len(x) <= 4:
        return None
    try:
        popt, pcov = curve_fit(
            model_4pl, x, y,
            p0=[np.mean(y[x == x.min()]), 1.0, np.median(x), np.mean(y[x == x.max()])],
            bounds=([-np.inf, 0.01, x.min() - 2, -np.inf],
                    [np.inf, 50, x.max() + 2, np.inf]),
            maxfev=30000,
        )
        predicted = model_4pl(x, *popt)
        ss_res = np.sum((y - predicted) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        r_squared = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
        log_ec50_se = np.sqrt(pcov[2, 2]) if np.isfinite(pcov[2, 2]) and pcov[2, 2] > 0 else np.nan
        if np.isfinite(log_ec50_se):
            t_crit = t_dist.ppf(0.975, df=len(x) - 4)
            ci_log = (popt[2] - t_crit * log_ec50_se, popt[2] + t_crit * log_ec50_se)
            ci = (10 ** ci_log[0], 10 ** ci_log[1])
        else:
            ci = (np.nan, np.nan)
        return {'popt': popt, 'pcov': pcov, 'ss_res': ss_res, 'r_squared': r_squared,
                'ec50': 10 ** popt[2], 'ec50_ci': ci}
    except Exception:
        return None


def f_test_vs_null(y, fit_result, alpha=0.01):
    y = np.asarray(y, float)
    y = y[np.isfinite(y)]
    if fit_result is None:
        return np.nan, np.nan, False
    ss_null = np.sum((y - y.mean()) ** 2)
    ss_4pl = fit_result['ss_res']
    df_4pl = len(y) - 4
    if df_4pl <= 0 or ss_4pl <= 0 or ss_null <= ss_4pl:
        return np.nan, 1.0, False
    f_stat = ((ss_null - ss_4pl) / 3) / (ss_4pl / df_4pl)
    p_value = 1 - f_dist.cdf(f_stat, 3, df_4pl)
    return f_stat, p_value, p_value < alpha


curve_results = {}
fit_summary_records = []
for condition, info in DATASETS.items():
    subset = crc_df[crc_df['Condition'].eq(condition)]
    x = subset['Concentration (log M)'].to_numpy(float)
    y = subset['Normalized Response'].to_numpy(float)
    result = fit_4pl(x, y)
    f_stat, p_value, significant = f_test_vs_null(y, result)
    if result is None:
        direction = 'fit failed'
        potency_in_range = False
    else:
        low_response = model_4pl(x.min(), *result['popt'])
        high_response = model_4pl(x.max(), *result['popt'])
        direction = 'increasing' if high_response > low_response else 'decreasing'
        potency_in_range = x.min() <= np.log10(result['ec50']) <= x.max()
    expected_direction = 'increasing' if info['mode'] == 'PAC-3310 antagonism' else 'decreasing'
    interpretable = significant and potency_in_range and direction == expected_direction
    if interpretable:
        interpretation = 'Interpretable 4PL'
    elif not significant:
        interpretation = '4PL not supported versus a flat response'
    elif not potency_in_range:
        interpretation = 'Candidate midpoint lies outside the tested range'
    else:
        interpretation = f'Curve direction is {direction}; expected {expected_direction}'
    curve_results[condition] = {
        'fit': result, 'significant': significant, 'interpretable': interpretable,
        'potency_in_range': potency_in_range, 'expected_direction': expected_direction,
        'p': p_value, 'F': f_stat,
    }
    fit_summary_records.append({
        'Condition': condition, 'Receptor': info['receptor'], 'Assay Mode': info['mode'],
        'Fixed Carbachol (nM)': info['fixed_carbachol_nm'],
        'Endpoint': subset['Normalized Endpoint'].iloc[0],
        'Model': '4PL' if interpretable else 'Points only',
        'EC50 (M)': result['ec50'] if result and interpretable else np.nan,
        'EC50 (µM)': result['ec50'] * 1e6 if result and interpretable else np.nan,
        'EC50 95% CI Low (M)': result['ec50_ci'][0] if result and interpretable else np.nan,
        'EC50 95% CI High (M)': result['ec50_ci'][1] if result and interpretable else np.nan,
        'Hill Slope': result['popt'][1] if result and interpretable else np.nan,
        'Low-concentration Asymptote': result['popt'][0] if result and interpretable else np.nan,
        'High-concentration Asymptote': result['popt'][3] if result and interpretable else np.nan,
        'R²': result['r_squared'] if result and interpretable else np.nan,
        'F Statistic': f_stat, 'F-test p': p_value, 'Curve Direction': direction,
        'Expected Direction': expected_direction,
        'Candidate logEC50': np.log10(result['ec50']) if result else np.nan,
        'Potency Within Tested Range': potency_in_range,
        'Significant at p<0.01': significant, 'Interpretation': interpretation,
    })

fit_summary_df = pd.DataFrame(fit_summary_records)
display(fit_summary_df)

## 9. Concentration-response Plots

In [ ]:
def plot_crc(condition, ax, show_legend=False):
    info = DATASETS[condition]
    subset = crc_df[crc_df['Condition'].eq(condition)]
    grouped = subset.groupby('Concentration (log M)')['Normalized Response']
    means, sems = grouped.mean().sort_index(), grouped.sem().sort_index()
    color = RECEPTOR_COLORS[info['receptor']]
    ax.errorbar(means.index, means.values, yerr=sems.values, fmt='o', color=color,
                markersize=5.5, elinewidth=1.3, markeredgecolor='white', markeredgewidth=0.5,
                label=info['receptor'], zorder=5)
    result = curve_results[condition]
    if result['interpretable']:
        x_fit = np.linspace(means.index.min(), means.index.max(), 250)
        ax.plot(x_fit, model_4pl(x_fit, *result['fit']['popt']), color=color, lw=2)
    ax.set_title(condition, fontsize=10.5, fontweight='bold')
    ax.set_xlabel(f"{info['compound']} concentration (log M)", fontsize=10)
    ax.set_ylabel(subset['Normalized Endpoint'].iloc[0], fontsize=10)
    ax.tick_params(labelsize=9)
    ax.grid(True, alpha=0.25)
    if show_legend:
        ax.legend(frameon=False)


fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, condition in zip(axes.flat, DATASETS):
    plot_crc(condition, ax)
fig.suptitle('PAC-3310 cAMP concentration-response analysis', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
crc_figure = FIGURE_DIR / 'PAC-3310_cAMP_CRCs.png'
plt.savefig(crc_figure, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {crc_figure}')

In [ ]:
mode_order = ['Carbachol agonism', 'PAC-3310 agonism', 'PAC-3310 antagonism']
panel_titles = ['Carbachol agonism', 'PAC-3310 agonism', 'PAC-3310 + carbachol antagonism']
fig, axes = plt.subplots(1, 3, figsize=(15, 4.7))

for ax, mode, title in zip(axes, mode_order, panel_titles):
    for receptor in ['M2', 'M4']:
        condition = next(k for k, v in DATASETS.items() if v['mode'] == mode and v['receptor'] == receptor)
        info = DATASETS[condition]
        subset = crc_df[crc_df['Condition'].eq(condition)]
        grouped = subset.groupby('Concentration (log M)')['Normalized Response']
        means, sems = grouped.mean().sort_index(), grouped.sem().sort_index()
        label = receptor
        if mode == 'PAC-3310 antagonism':
            label = f"{receptor} ({int(info['fixed_carbachol_nm'])} nM carbachol)"
        ax.errorbar(means.index, means.values, yerr=sems.values, fmt='o',
                    color=RECEPTOR_COLORS[receptor], markersize=5.5, elinewidth=1.3,
                    markeredgecolor='white', markeredgewidth=0.5, label=label, zorder=5)
        result = curve_results[condition]
        if result['interpretable']:
            x_fit = np.linspace(means.index.min(), means.index.max(), 250)
            ax.plot(x_fit, model_4pl(x_fit, *result['fit']['popt']),
                    color=RECEPTOR_COLORS[receptor], lw=2)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Concentration (log M)', fontsize=LABEL_SIZE)
    ylabel = '% Carbachol Signaling Inhibition' if mode == 'PAC-3310 antagonism' else 'cAMP (% Vehicle Control)'
    ax.set_ylabel(ylabel, fontsize=LABEL_SIZE)
    ax.tick_params(labelsize=TICK_SIZE)
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=False, fontsize=LEGEND_SIZE)

plt.tight_layout()
publication_figure = FIGURE_DIR / 'PAC-3310_cAMP_publication_panels.png'
plt.savefig(publication_figure, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {publication_figure}')

## 10. Export Analysis Tables

In [ ]:
replicate_export = replicate_df.copy()
replicate_export.to_csv(PROCESSED_DIR / 'PAC-3310_cAMP_replicate_results.csv', index=False)
group_summary_df.to_csv(PROCESSED_DIR / 'PAC-3310_cAMP_group_summary.csv', index=False)
control_summary_df.to_csv(PROCESSED_DIR / 'PAC-3310_cAMP_control_summary.csv', index=False)
standard_summary_df.to_csv(PROCESSED_DIR / 'PAC-3310_cAMP_standard_curve_summary.csv', index=False)
fit_summary_df.to_csv(PROCESSED_DIR / 'PAC-3310_cAMP_fit_summary.csv', index=False)

print('Exported:')
for path in sorted(PROCESSED_DIR.glob('PAC-3310_cAMP_*.csv')):
    print(f'  {path.name}')